In [5]:
import os
import pandas as pd

from src.utils import get_base_dir, save_to_pickle, load_from_pickle

In [2]:
dataset = "dooway"
region = "Basilicata" # "EmiliaRomagna"
region_str = region.lower() # "emilia_romagna"
base_dir = os.path.join(get_base_dir(), region)
full_data_dir = os.path.join(base_dir, "original_data")
original_dir = os.path.join(base_dir, "heterodata")
subset_dir = os.path.join(base_dir, "subset", "heterodata")

In [3]:
df_gt = pd.read_parquet(os.path.join(full_data_dir, "user_labels.parquet"))

df_gt.head()

,unique_id_user,label,label_id
0,56,genZ,11
1,464,family_oriented,9
2,624,family_oriented,9
3,735,lazy,12
4,1016,cultured,3


In [4]:
df_gt_sorted = df_gt.sort_values(by="unique_id_user").reset_index(drop=True)

df_gt_sorted.tail()

,unique_id_user,label,label_id
3538,589975,curious,4
3539,590156,family_oriented,9
3540,590216,family_oriented,9
3541,590453,curious,4
3542,590706,genZ,11


In [5]:
mapping_U = df_gt_sorted["unique_id_user"].to_dict()
mapping_U_rev = {v: k for k, v in mapping_U.items()}


In [6]:
df_gt_final = df_gt_sorted.copy()
df_gt_final["unique_id_user"] = df_gt_final["unique_id_user"].map(mapping_U_rev)
#df_gt_final.to_parquet(os.path.join(subset_dir, "user_labels.parquet"), index=False)

df_gt_final.head()

,unique_id_user,label,label_id
0,0,genZ,11
1,1,family_oriented,9
2,2,family_oriented,9
3,3,lazy,12
4,4,cultured,3


In [7]:
users_ok = df_gt["unique_id_user"].drop_duplicates().tolist()
print(f"Number of users with ground truth: {len(users_ok)}")

Number of users with ground truth: 3543


In [12]:
# SUBSET USERS

df_U = pd.read_parquet(os.path.join(original_dir, "features", "dataframes", "user.parquet"))
df_U_sub = df_U[df_U["unique_id"].isin(users_ok)].reset_index(drop=True)
df_U_sub["unique_id"] = df_U_sub["unique_id"].map(mapping_U_rev)
df_U_sub.to_parquet(os.path.join(subset_dir, "features", "dataframes", "user.parquet"), index=False)

df_U_sub.head()


,unique_id,profile_url_encoded
0,0,"[-0.15778091549873352, -0.05948127806186676, 0..."
1,1,"[-0.13735520839691162, -0.15449145436286926, -..."
2,2,"[0.14104053378105164, -0.11732335388660431, -0..."
3,3,"[-0.012760162353515625, -0.30527496337890625, ..."
4,4,"[0.06612680852413177, -0.06949793547391891, -0..."


In [8]:
#SUBSET POI

df_P = pd.read_parquet(os.path.join(original_dir, "features", "dataframes", "poi.parquet"))
df_P.head()

,rank_absolute,total_photos,latitude,longitude,is_claimed,unique_id,main_image_embedding,restaurant,hotel,attraction,...,title_embedding,free_parking,paid_parking,reservation,excursion,spa,vegetarian_cuisine,vegan_cuisine,gluten-free_cuisine,lactose-free_cuisine
0,1,1,40.508039,15.812844,False,0,"[-0.19753423, 0.22135854, 0.1589038, -0.080560...",True,False,False,...,"[-0.0924561396241188, 0.3570820093154907, -0.1...",True,True,True,True,True,True,True,True,True
1,2,427,40.522884,15.785847,True,1,"[0.46170676, 0.29732698, 0.17882484, -0.351835...",True,False,False,...,"[0.03634057939052582, 0.37386876344680786, 0.0...",True,False,False,False,False,False,False,False,False
2,3,4,40.508718,15.811278,True,2,"[-0.031655762, 0.32317862, -0.25760216, 0.0752...",True,False,False,...,"[0.2866106331348419, 0.33424651622772217, -0.0...",True,True,True,True,True,True,True,True,True
3,4,4,40.507664,15.813661,True,3,"[-0.14220355, -0.040702403, -0.09907015, -0.24...",True,False,False,...,"[0.6681708693504333, -0.4947139322757721, -0.1...",True,True,True,True,True,True,True,True,True
4,5,93,40.510964,15.806366,False,4,"[-0.109207876, 0.08907696, -0.011386456, -0.04...",True,False,False,...,"[0.4959424138069153, -0.09553607553243637, -0....",True,True,True,True,True,True,True,True,True


In [5]:
df_P.shape

(14853, 27)

In [9]:
df_UP = pd.read_parquet(os.path.join(full_data_dir, "edges", "user_visited_poi.parquet"))
df_UP_ok = df_UP[df_UP["unique_id_user"].isin(users_ok)]
poi_ok = df_UP_ok["unique_id_poi"].drop_duplicates().tolist()

df_poi_sub = df_P[df_P["unique_id"].isin(poi_ok)]
df_poi_sub_sorted = df_poi_sub.sort_values(by="unique_id").reset_index(drop=True)

df_poi_sub_sorted.tail()




,rank_absolute,total_photos,latitude,longitude,is_claimed,unique_id,main_image_embedding,restaurant,hotel,attraction,...,title_embedding,free_parking,paid_parking,reservation,excursion,spa,vegetarian_cuisine,vegan_cuisine,gluten-free_cuisine,lactose-free_cuisine
4590,5,33,40.601530,16.383566,False,14771,"[0.13745779, 0.037608325, -0.46893758, 0.24494...",False,False,True,...,"[0.3067038655281067, 0.18494658172130585, -0.2...",True,True,True,True,True,True,True,True,True
4591,95,43,40.666750,16.606743,False,14776,"[0.18945308, -0.16489324, -0.35077715, -0.2888...",False,False,True,...,"[0.05418763682246208, 0.30174461007118225, 0.1...",True,True,True,True,True,True,True,True,True
4592,63,4,40.665297,16.606574,False,14777,"[0.14391264, 0.34635627, -0.5263301, -0.393687...",False,False,True,...,"[0.05142820626497269, 0.5848613381385803, -0.0...",True,True,True,True,True,True,True,True,True
4593,41,2,40.663663,16.597749,True,14781,"[0.35685343, 0.423259, -0.2659947, 0.14794141,...",False,False,True,...,"[0.2838599681854248, -0.037209201604127884, -0...",True,True,True,True,True,True,True,True,True
4594,11,245,40.250446,16.475908,True,14819,"[-0.04509115, -0.048610568, -0.006292559, 0.36...",False,False,True,...,"[0.396616667509079, 0.3997695744037628, -0.115...",True,True,True,True,True,True,True,True,True


In [10]:
mapping_P = df_poi_sub_sorted["unique_id"].to_dict()
mapping_P_rev = {v: k for k, v in mapping_P.items()}

In [15]:
df_poi_final = df_poi_sub_sorted.copy()
df_poi_final["unique_id"] = df_poi_final["unique_id"].map(mapping_P_rev)
df_poi_final.drop(["main_image_embedding"], axis=1, inplace=True)
df_poi_final.to_parquet(os.path.join(subset_dir, "features", "dataframes", "poi.parquet"), index=False)

df_poi_final.head()

,rank_absolute,total_photos,latitude,longitude,is_claimed,unique_id,restaurant,hotel,attraction,avg_rating,...,title_embedding,free_parking,paid_parking,reservation,excursion,spa,vegetarian_cuisine,vegan_cuisine,gluten-free_cuisine,lactose-free_cuisine
0,1,1,40.508039,15.812844,False,0,True,False,False,4.6,...,"[-0.0924561396241188, 0.3570820093154907, -0.1...",True,True,True,True,True,True,True,True,True
1,2,427,40.522884,15.785847,True,1,True,False,False,4.7,...,"[0.03634057939052582, 0.37386876344680786, 0.0...",True,False,False,False,False,False,False,False,False
2,4,4,40.507664,15.813661,True,2,True,False,False,4.6,...,"[0.6681708693504333, -0.4947139322757721, -0.1...",True,True,True,True,True,True,True,True,True
3,5,93,40.510964,15.806366,False,3,True,False,False,3.8,...,"[0.4959424138069153, -0.09553607553243637, -0....",True,True,True,True,True,True,True,True,True
4,2,2,40.507336,15.816956,False,4,True,False,False,3.8,...,"[-0.10648553818464279, -0.07584365457296371, 0...",True,True,True,True,True,True,True,True,True


In [13]:
df_poi_final.shape

(4595, 26)

In [14]:
df_UP_ok.head()

,unique_id_user,unique_id_poi
0,182036,13976
1,583934,13976
2,356356,13976
3,451564,13976
4,460503,13976


In [16]:
df_UP_ok["unique_id_user"] = df_UP_ok["unique_id_user"].map(mapping_U_rev)
df_UP_ok["unique_id_poi"] = df_UP_ok["unique_id_poi"].map(mapping_P_rev)
#df_UP_ok.to_parquet(os.path.join(subset_dir, "edgelists", "dataframes", "user_visited_poi.parquet"), index=False)

df_UP_ok.head()

/tmp/ipykernel_841431/3334602716.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_UP_ok["unique_id_user"] = df_UP_ok["unique_id_user"].map(mapping_U_rev)
/tmp/ipykernel_841431/3334602716.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_UP_ok["unique_id_poi"] = df_UP_ok["unique_id_poi"].map(mapping_P_rev)


,unique_id_user,unique_id_poi
0,1094,4528
1,3508,4528
2,2174,4528
3,2743,4528
4,2812,4528


In [17]:
# SUBSET IMAGES 

df_I = pd.read_parquet(os.path.join(original_dir, "features", "dataframes", "image.parquet"))
df_I.head()

,unique_id,main_image_embedding
0,0,"[-0.19753423, 0.22135854, 0.1589038, -0.080560..."
1,1,"[0.46170676, 0.29732698, 0.17882484, -0.351835..."
2,2,"[-0.031655762, 0.32317862, -0.25760216, 0.0752..."
3,3,"[-0.14220355, -0.040702403, -0.09907015, -0.24..."
4,4,"[-0.109207876, 0.08907696, -0.011386456, -0.04..."


In [18]:
df_I_sub = df_I[df_I["unique_id"].isin(poi_ok)]
df_I_sub_sorted = df_I_sub.sort_values(by="unique_id").reset_index(drop=True)

df_I_sub_sorted.tail()

,unique_id,main_image_embedding
4590,14771,"[0.13745779, 0.037608325, -0.46893758, 0.24494..."
4591,14776,"[0.18945308, -0.16489324, -0.35077715, -0.2888..."
4592,14777,"[0.14391264, 0.34635627, -0.5263301, -0.393687..."
4593,14781,"[0.35685343, 0.423259, -0.2659947, 0.14794141,..."
4594,14819,"[-0.04509115, -0.048610568, -0.006292559, 0.36..."


In [19]:
df_I_final = df_I_sub_sorted.copy()
df_I_final["unique_id"] = df_I_final["unique_id"].map(mapping_P_rev)
df_I_final.to_parquet(os.path.join(subset_dir, "features", "dataframes", "image.parquet"), index=False)

df_I_final.head()

,unique_id,main_image_embedding
0,0,"[-0.19753423, 0.22135854, 0.1589038, -0.080560..."
1,1,"[0.46170676, 0.29732698, 0.17882484, -0.351835..."
2,2,"[-0.14220355, -0.040702403, -0.09907015, -0.24..."
3,3,"[-0.109207876, 0.08907696, -0.011386456, -0.04..."
4,4,"[-0.23309213, 0.26094317, 0.0034338012, -0.168..."


In [24]:
df_IP = df_I[["unique_id"]].copy()
df_IP.rename(columns={"unique_id": "unique_id_poi"}, inplace=True)
df_IP["unique_id_image"] = df_IP["unique_id_poi"]
df_IP.to_parquet(os.path.join(subset_dir, "edgelists", "dataframes", "poi_has_main_image_image.parquet"), index=False)

df_IP.head()

,unique_id_poi,unique_id_image
0,0,0
1,1,1
2,2,2
3,3,3
4,4,4


In [17]:
#SUBSET REVIEW

df_R = pd.read_parquet(os.path.join(original_dir, "features", "dataframes", "review.parquet"))
df_R.head()

,timestamp,rating_value,local_guide,unique_id,review_text_embedding,positive_quality,negative_quality,noise_level,group_size,waiting_time,description_embedding
0,2020-12-04 10:39:16+00:00,4.0,True,0,"[-0.02132909931242466, 0.0855952724814415, 0.0...",False,False,-1,-1,-1,"[0.0973015590350905, 0.048725716206544194, 0.0..."
1,2021-12-04 10:39:16+00:00,5.0,True,1,"[0.11749532073736191, 0.027359535917639732, 0....",False,False,-1,-1,-1,"[0.0973015590350905, 0.048725716206544194, 0.0..."
2,2022-12-04 10:39:16+00:00,5.0,True,2,"[-0.0026724785566329956, 0.1681678593158722, 0...",False,False,-1,-1,-1,"[0.0973015590350905, 0.048725716206544194, 0.0..."
3,2021-12-04 10:39:16+00:00,5.0,True,3,"[0.11633721739053726, -0.0018988546216860414, ...",False,False,-1,-1,-1,"[0.0973015590350905, 0.048725716206544194, 0.0..."
4,2021-12-04 10:39:16+00:00,4.0,False,4,"[-0.1472267359495163, -0.14695429801940918, 0....",False,False,-1,-1,-1,"[0.0973015590350905, 0.048725716206544194, 0.0..."


In [18]:
df_RP = pd.read_parquet(os.path.join(full_data_dir, "edges", "review_is_about_poi.parquet"))
df_RP_ok = df_RP[df_RP["unique_id_poi"].isin(poi_ok)]
review_ok = df_RP_ok["unique_id_review"].drop_duplicates().tolist()

df_R_sub = df_R[df_R["unique_id"].isin(review_ok)]
df_R_sub_sorted = df_R_sub.sort_values(by="unique_id").reset_index(drop=True)

df_R_sub_sorted.tail()

,timestamp,rating_value,local_guide,unique_id,review_text_embedding,positive_quality,negative_quality,noise_level,group_size,waiting_time,description_embedding
1597210,2024-01-13 10:29:33+00:00,5.0,False,1700362,"[0.12187313990745491, -0.024273028586362592, 0...",False,False,-1,-1,-1,"[0.0973015590350905, 0.048725716206544194, 0.0..."
1597211,2024-01-13 10:29:33+00:00,5.0,True,1700363,"[0.12187313990745491, -0.024273028586362592, 0...",False,False,-1,-1,-1,"[0.0973015590350905, 0.048725716206544194, 0.0..."
1597212,2024-01-13 10:29:33+00:00,5.0,False,1700364,"[0.12187313990745491, -0.024273028586362592, 0...",False,False,-1,-1,-1,"[0.0973015590350905, 0.048725716206544194, 0.0..."
1597213,2024-01-13 10:29:33+00:00,5.0,False,1700365,"[0.12187313990745491, -0.024273028586362592, 0...",False,False,-1,-1,-1,"[0.0973015590350905, 0.048725716206544194, 0.0..."
1597214,2024-01-13 10:29:33+00:00,5.0,False,1700366,"[0.12187313990745491, -0.024273028586362592, 0...",False,False,-1,-1,-1,"[0.0973015590350905, 0.048725716206544194, 0.0..."


In [19]:
mapping_R = df_R_sub_sorted["unique_id"].to_dict()
mapping_R_rev = {v: k for k, v in mapping_R.items()}

In [20]:
df_R_final = df_R_sub_sorted.copy()
df_R_final["unique_id"] = df_R_final["unique_id"].map(mapping_R_rev)
df_R_final.to_parquet(os.path.join(subset_dir, "features", "dataframes", "review.parquet"), index=False)

df_R_final.head()

,timestamp,rating_value,local_guide,unique_id,review_text_embedding,positive_quality,negative_quality,noise_level,group_size,waiting_time,description_embedding
0,2020-12-04 10:39:16+00:00,4.0,True,0,"[-0.02132909931242466, 0.0855952724814415, 0.0...",False,False,-1,-1,-1,"[0.0973015590350905, 0.048725716206544194, 0.0..."
1,2021-12-04 10:39:16+00:00,5.0,True,1,"[0.11749532073736191, 0.027359535917639732, 0....",False,False,-1,-1,-1,"[0.0973015590350905, 0.048725716206544194, 0.0..."
2,2022-12-04 10:39:16+00:00,5.0,True,2,"[-0.0026724785566329956, 0.1681678593158722, 0...",False,False,-1,-1,-1,"[0.0973015590350905, 0.048725716206544194, 0.0..."
3,2021-12-04 10:39:16+00:00,5.0,True,3,"[0.11633721739053726, -0.0018988546216860414, ...",False,False,-1,-1,-1,"[0.0973015590350905, 0.048725716206544194, 0.0..."
4,2021-12-04 10:39:16+00:00,4.0,False,4,"[-0.1472267359495163, -0.14695429801940918, 0....",False,False,-1,-1,-1,"[0.0973015590350905, 0.048725716206544194, 0.0..."


In [31]:
df_RP_ok["unique_id_review"] = df_RP_ok["unique_id_review"].map(mapping_R_rev)
df_RP_ok["unique_id_poi"] = df_RP_ok["unique_id_poi"].map(mapping_P_rev)
df_RP_ok.to_parquet(os.path.join(subset_dir, "edgelists", "dataframes", "review_is_about_poi.parquet"), index=False)

df_RP_ok.head()

/tmp/ipykernel_652714/4163126439.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_RP_ok["unique_id_review"] = df_RP_ok["unique_id_review"].map(mapping_R_rev)
/tmp/ipykernel_652714/4163126439.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_RP_ok["unique_id_poi"] = df_RP_ok["unique_id_poi"].map(mapping_P_rev)


,unique_id_review,unique_id_poi
0,0,4528
1,1,4528
2,2,4528
3,3,4528
4,4,4528


In [32]:
df_UR = pd.read_parquet(os.path.join(full_data_dir, "edges", "user_wrote_review.parquet"))
df_UR_ok = df_UR[df_UR["unique_id_user"].isin(users_ok) & df_UR["unique_id_review"].isin(review_ok)]

df_UR_ok["unique_id_user"] = df_UR_ok["unique_id_user"].map(mapping_U_rev)
df_UR_ok["unique_id_review"] = df_UR_ok["unique_id_review"].map(mapping_R_rev)
df_UR_ok.to_parquet(os.path.join(subset_dir, "edgelists", "dataframes", "user_wrote_review.parquet"), index=False)

df_UR_ok.head()

/tmp/ipykernel_652714/505038718.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_UR_ok["unique_id_user"] = df_UR_ok["unique_id_user"].map(mapping_U_rev)
/tmp/ipykernel_652714/505038718.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_UR_ok["unique_id_review"] = df_UR_ok["unique_id_review"].map(mapping_R_rev)


,unique_id_user,unique_id_review
0,1094,0
1,3508,1
2,2174,2
3,2743,3
4,2812,4


In [22]:
# ZONE

df_PZ = pd.read_parquet(os.path.join(full_data_dir, "edges", "poi_is_in_zone.parquet"))
df_PZ_ok = df_PZ[df_PZ["unique_id_poi"].isin(poi_ok)]

df_PZ_ok["unique_id_poi"] = df_PZ_ok["unique_id_poi"].map(mapping_P_rev)
df_PZ_ok.to_parquet(os.path.join(subset_dir, "edgelists", "dataframes", "poi_is_in_zone.parquet"), index=False)

df_PZ_ok.head()

/tmp/ipykernel_711702/1775885287.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_PZ_ok["unique_id_poi"] = df_PZ_ok["unique_id_poi"].map(mapping_P_rev)


,unique_id_poi,unique_id_zone
0,4528,0
40,4358,0
41,4175,0
47,3174,0
75,4590,0


In [24]:
df_Z = pd.read_parquet(os.path.join(original_dir, "features", "dataframes", "zone.parquet"))
df_Z.to_parquet(os.path.join(subset_dir, "features", "dataframes", "zone.parquet"), index=False)
df_Z

,unique_id,zone
0,0,9053426


In [26]:
# CATEGORY

df_C = pd.read_parquet(os.path.join(original_dir, "features", "dataframes", "category.parquet"))
df_C.head()

,unique_id,category_encoded
0,0,"[-0.3043419420719147, -0.2923114597797394, 0.1..."
1,1,"[-0.1025187149643898, 0.13989366590976715, 0.0..."
2,2,"[-0.13989470899105072, -0.16000115871429443, 0..."
3,3,"[0.1511448621749878, 0.26360148191452026, -0.0..."
4,4,"[-0.11031092703342438, -0.12447044998407364, 0..."


In [36]:
df_PC = pd.read_parquet(os.path.join(full_data_dir, "edges", "poi_is_of_category.parquet"))
df_PC_ok = df_PC[df_PC["unique_id_poi"].isin(poi_ok)]
category_ok = df_PC_ok["unique_id_category"].drop_duplicates().tolist()

df_C_sub = df_C[df_C["unique_id"].isin(category_ok)]
df_C_sub_sorted = df_C_sub.sort_values(by="unique_id").reset_index(drop=True)

df_C_sub_sorted.tail()

,unique_id,category_encoded
543,1516,"[-0.10139630734920502, -0.1752546727657318, 0...."
544,1518,"[-0.04530372843146324, -0.15113785862922668, 0..."
545,1520,"[-0.0028661040123552084, -0.03507000207901001,..."
546,1521,"[-0.041608281433582306, -0.33607247471809387, ..."
547,1522,"[0.10363786667585373, -0.06715843826532364, -0..."


In [37]:
mapping_C = df_C_sub_sorted["unique_id"].to_dict()
mapping_C_rev = {v: k for k, v in mapping_C.items()}


In [38]:
df_C_final = df_C_sub_sorted.copy()
df_C_final["unique_id"] = df_C_final["unique_id"].map(mapping_C_rev)
df_C_final.to_parquet(os.path.join(subset_dir, "features", "dataframes", "category.parquet"), index=False)

df_C_final.head()

,unique_id,category_encoded
0,0,"[-0.3043419420719147, -0.2923114597797394, 0.1..."
1,1,"[-0.13989470899105072, -0.16000115871429443, 0..."
2,2,"[-0.11031092703342438, -0.12447044998407364, 0..."
3,3,"[-0.20324835181236267, 0.16588371992111206, -0..."
4,4,"[0.24359354376792908, 0.06765929609537125, -0...."


In [39]:
df_PC_ok["unique_id_poi"] = df_PC_ok["unique_id_poi"].map(mapping_P_rev)
df_PC_ok["unique_id_category"] = df_PC_ok["unique_id_category"].map(mapping_C_rev)
df_PC_ok.to_parquet(os.path.join(subset_dir, "edgelists", "dataframes", "poi_is_of_category.parquet"), index=False)

df_PC_ok.head()

/tmp/ipykernel_711702/1897962195.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_PC_ok["unique_id_poi"] = df_PC_ok["unique_id_poi"].map(mapping_P_rev)
/tmp/ipykernel_711702/1897962195.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_PC_ok["unique_id_category"] = df_PC_ok["unique_id_category"].map(mapping_C_rev)


,unique_id_poi,unique_id_category
0,0,9
1,1,546
3,2,329
4,3,546
5,3,135


In [53]:
# SERVICE

df_S = pd.read_parquet(os.path.join(original_dir, "features", "dataframes", "service.parquet"))
df_S

,unique_id,name_embedding
0,0,"[-0.26262152194976807, 0.1820594072341919, -0...."
1,1,"[-0.11518297344446182, -0.14848442375659943, -..."
2,2,"[0.3130197525024414, 0.2622040808200836, 0.189..."
3,3,"[0.05102507397532463, -0.03212948143482208, -0..."
4,4,"[0.5007088780403137, 0.10016030818223953, -0.0..."
5,5,"[0.24551928043365479, 0.034091830253601074, -0..."


In [51]:
df_RS = pd.read_parquet(os.path.join(original_dir, "edgelists", "review_is_about_service.parquet"))
df_RS_ok = df_RS[df_RS["review_id"].isin(review_ok)]
service_ok = df_RS_ok["service_id"].drop_duplicates().tolist()

df_S_sub = df_S[df_S["unique_id"].isin(service_ok)]
df_S_sub_sorted = df_S_sub.sort_values(by="unique_id").reset_index(drop=True)

df_S_sub_sorted

,unique_id,name_embedding
0,0,"[-0.26262152194976807, 0.1820594072341919, -0...."
1,1,"[-0.11518297344446182, -0.14848442375659943, -..."
2,2,"[0.3130197525024414, 0.2622040808200836, 0.189..."
3,3,"[0.05102507397532463, -0.03212948143482208, -0..."
4,4,"[0.5007088780403137, 0.10016030818223953, -0.0..."
5,5,"[0.24551928043365479, 0.034091830253601074, -0..."


In [54]:
mapping_S = df_S_sub_sorted["unique_id"].to_dict()
mapping_S_rev = {v: k for k, v in mapping_S.items()}

In [55]:
df_S_final = df_S_sub_sorted.copy()
df_S_final["unique_id"] = df_S_final["unique_id"].map(mapping_S_rev)
df_S_final.to_parquet(os.path.join(subset_dir, "features", "dataframes", "service.parquet"), index=False)

df_S_final

,unique_id,name_embedding
0,0,"[-0.26262152194976807, 0.1820594072341919, -0...."
1,1,"[-0.11518297344446182, -0.14848442375659943, -..."
2,2,"[0.3130197525024414, 0.2622040808200836, 0.189..."
3,3,"[0.05102507397532463, -0.03212948143482208, -0..."
4,4,"[0.5007088780403137, 0.10016030818223953, -0.0..."
5,5,"[0.24551928043365479, 0.034091830253601074, -0..."


In [56]:
df_RS_ok.head()

,review_id,service_id,score
0,653221,0,5.0
1,653221,1,5.0
2,653221,2,5.0
3,653224,0,5.0
4,653224,1,5.0


In [57]:
df_RS_ok["review_id"] = df_RS_ok["review_id"].map(mapping_R_rev)
df_RS_ok["service_id"] = df_RS_ok["service_id"].map(mapping_S_rev)
df_RS_ok.to_parquet(os.path.join(subset_dir, "edgelists", "dataframes", "review_is_about_service.parquet"), index=False)

df_RS_ok.head()

/tmp/ipykernel_711702/3545636465.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_RS_ok["review_id"] = df_RS_ok["review_id"].map(mapping_R_rev)
/tmp/ipykernel_711702/3545636465.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_RS_ok["service_id"] = df_RS_ok["service_id"].map(mapping_S_rev)


,review_id,service_id,score
0,613946,0,5.0
1,613946,1,5.0
2,613946,2,5.0
3,613949,0,5.0
4,613949,1,5.0


In [62]:
mapping = {
    "category": mapping_C,
    "poi": mapping_P,
    "service": mapping_S,
    "review": mapping_R,
    "user": mapping_U,   
}

save_to_pickle(mapping, os.path.join(base_dir, "subset", "mapping.pkl"))

In [8]:
#TODO meglio: subset poi-image
import torch

num = torch.load(os.path.join(subset_dir, "features", "tensors", "image.pt")).shape[0]
print(num)
PI = torch.load(os.path.join(subset_dir, "edgelists", "tensors", "poi_has_main_image_image.pt"))
PI_sub = PI[:, :num]
print(PI_sub.shape)
torch.save(PI_sub, os.path.join(subset_dir, "edgelists", "tensors", "poi_has_main_image_image.pt"))
IP = torch.load(os.path.join(subset_dir, "edgelists", "tensors", "image_is_main_image_of_poi.pt"))
IP_sub = IP[:, :num]
print(IP_sub.shape)
torch.save(IP_sub, os.path.join(subset_dir, "edgelists", "tensors", "image_is_main_image_of_poi.pt"))


4595
torch.Size([2, 4595])
torch.Size([2, 4595])


In [10]:
#Ground-truth multilabel
#full_data_dir = os.path.join(base_dir, "original_data")
#original_dir = os.path.join(base_dir, "heterodata")
#subset_dir = os.path.join(base_dir, "subset", "heterodata")

#Step 1: recuperare coppie review-labels da full_data_dir/review_llm.parquet (id di review parte da 0 con solo casi not NaN)
df_gt = pd.read_parquet(os.path.join(full_data_dir, "review_llm.parquet")) #unique_id

#Step 2: risalire da review a utente sfruttando la relazione user_writes_review
df_UR = pd.read_parquet(os.path.join(full_data_dir, "edges", "user_wrote_review.parquet")) #unique_id_user, unique_id_review

df_gt_users = df_gt.merge(
    df_UR,
    left_on="unique_id",
    right_on="unique_id_review",
    how="left"
)

df_gt_users = df_gt_users.drop(columns=["unique_id", "unique_id_review"])
df_gt_users = df_gt_users.rename(columns={"unique_id_user": "user_id"})
df_gt_users.to_parquet(os.path.join(full_data_dir, "users_all_multilabel.parquet"), index=False)

df_gt_users_merged = (
    df_gt_users.groupby("user_id", as_index=False)
    .mean(numeric_only=True)
    .round(1)
)
df_gt_users_merged.to_parquet(os.path.join(full_data_dir, "users_avg_multilabel.parquet"), index=False)

#Step 3: verificare subset e fare il remapping 
mapping_U = load_from_pickle(os.path.join(base_dir, "subset", "mapping.pkl"))["user"] # mapping_U = {new_id: old_id}
# We need inverse mapping (old_id -> new_id)
inverse_mapping = {old: new for new, old in mapping_U.items()}

df_gt_users_remapped = df_gt_users.copy()
df_gt_users_remapped["user_id"] = df_gt_users_remapped["user_id"].map(inverse_mapping)
df_gt_users_merged.to_parquet(os.path.join(base_dir, "subset", "users_all_multilabel.parquet"), index=False)

df_gt_users_merged_remapped = df_gt_users_merged.copy()
df_gt_users_merged_remapped["user_id"] = df_gt_users_merged_remapped["user_id"].map(inverse_mapping)
df_gt_users_merged_remapped = df_gt_users_merged_remapped.rename(columns={"user_id": "unique_id_user"})
df_gt_users_merged_remapped = df_gt_users_merged_remapped.fillna(0.0)
df_gt_users_merged_remapped.to_parquet(os.path.join(base_dir, "subset", "users_avg_multilabel.parquet"), index=False)


In [13]:
test = pd.read_parquet(os.path.join(subset_dir, "user_labels.parquet"))
test.head()

,unique_id_user,label_id
0,0,11
1,1,9
2,2,9
3,3,12
4,4,3


In [18]:
test.shape

(3543, 2)